# **ATRIUM WP 4.1.3 Workflow Demo [WORK IN PROGRESS]**

### **1. Entity Extraction**

#### Module Initialisation

In [ ]:
import spacy
import srsly
from tqdm import tqdm

# Setup module paths
m_ner_model_path = "en_deberta_v3_base_ner_method"
a_ner_model_path = "en_deberta_v3_base_ner_activity"
g_ner_model_path = "en_deberta_v3_base_ner_goal"

# Setup input and output paths
ee_input_path = "./Dataset/example_subset_10.jsonl"
ee_output_path = "./Dataset/example_subset_10_EE.jsonl"

#### Module Functions

In [ ]:
def NER(model_path, in_data, entity):
    ner_model = spacy.load(model_path)
    annotated_data = []
    for row in tqdm(in_data, desc=f"NER for {entity}"):
        sent_nlp = ner_model(row["text"])
        ner_spans = [{"start_char": span.start_char, "end_char": span.end_char, "token_start":span.start, "token_end":span.end, "mention":row["text"][span.start_char:span.end_char], "label": entity} for span in sent_nlp.ents]
        if "spans" in row:
            row["spans"] += ner_spans
        else:
            row["spans"] = ner_spans

        row["_annotator_id"] = "NER"
        row["_session_id"] = "NER"
        annotated_data.append(row)

    return(annotated_data)

#### Module Call

In [ ]:
input_data = srsly.read_jsonl(ee_input_path)

sents_with_M = NER(m_ner_model_path, input_data, "METHOD")
sents_with_A = NER(a_ner_model_path, sents_with_M, "ACTIVITY")
sents_with_G = NER(g_ner_model_path, sents_with_A, "GOAL")

srsly.write_jsonl(ee_output_path, sents_with_G)